# Part 2 — Preprocessing Audio: DAIC-WOZ
**Pipeline**: Klasifikasi Kesehatan Mental Berbasis Audio
**Peran**: ML & Data Engineer

## Analisis File Metadata
- **TRANSCRIPT.csv**: Digunakan untuk Speaker Diarization → filter segmen "Participant" saja
- **COVAREP.csv**: Kolom index-1 adalah flag VUV (Voiced=1, Unvoiced=0) → digunakan sebagai VAD mask
- **FORMANT.csv**: Berisi F1-F5 formant frequencies → TIDAK dipakai di preprocessing, relevan untuk feature extraction

## Tahapan Pipeline
1. Speaker Diarization via TRANSCRIPT.csv (filter suara Participant)
2. VAD via COVAREP VUV column (lebih akurat dari energy-based VAD)
3. Fallback ke librosa VAD jika COVAREP tidak tersedia
4. Resampling ke 16kHz mono
5. Noise Reduction (noisereduce)
6. Amplitude Normalization ke [-1, 1]
7. Simpan ke data/cleaned/{participant_id}.wav


In [13]:
import os
import sys
import logging
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import noisereduce as nr
from tqdm import tqdm

warnings.filterwarnings('ignore')

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')


In [14]:
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.path.abspath(os.getcwd())

# Script ada di experiments/, naik 1 level ke root project menthealth-ai/
BASE_DIR      = os.path.dirname(current_dir)
DAIC_DIR      = os.path.join(BASE_DIR, 'data', 'raw', 'DAIC-WOZ')
CLEANED_DIR   = os.path.join(BASE_DIR, 'data', 'cleaned')
LOG_DIR       = os.path.join(BASE_DIR, 'data', 'processed')

os.makedirs(CLEANED_DIR, exist_ok=True)
os.makedirs(LOG_DIR,     exist_ok=True)

TARGET_SR      = 16000   # Hz — standar speech processing
MIN_SEG_DUR    = 0.3     # detik — buang segmen terlalu pendek
TOP_DB_VAD     = 25      # dB threshold untuk librosa VAD (fallback)
COVAREP_VUV_COL = 1      # index kolom VUV di COVAREP.csv (0-indexed)
# COVAREP frame rate = 100 fps (10ms per frame), sama dengan default librosa hop


In [15]:
logging.basicConfig(
    filename=os.path.join(LOG_DIR, 'preprocessing.log'),
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
log = logging.getLogger(__name__)
log.addHandler(logging.StreamHandler(sys.stdout))

print(f"DAIC-WOZ DIR : {DAIC_DIR}")
print(f"Cleaned DIR  : {CLEANED_DIR}")


DAIC-WOZ DIR : d:\repositories\menthealth-ai\data\raw\DAIC-WOZ
Cleaned DIR  : d:\repositories\menthealth-ai\data\cleaned


In [16]:
def get_participant_segments(transcript_path):
    """
    Membaca TRANSCRIPT.csv dan mengembalikan list (start_sec, stop_sec)
    hanya untuk baris speaker == 'Participant'.
    Format file: start_time [TAB] stop_time [TAB] speaker [TAB] value
    """
    try:
        df = pd.read_csv(transcript_path, sep='\t')
        df.columns = [c.strip().lower() for c in df.columns]
        df_p = df[df['speaker'].str.strip().str.lower() == 'participant'].copy()
        segments = list(zip(
            df_p['start_time'].astype(float),
            df_p['stop_time'].astype(float)
        ))
        return segments
    except Exception as e:
        log.warning(f"Gagal baca TRANSCRIPT: {e}")
        return []

def extract_segments_audio(y, sr, segments, min_dur=MIN_SEG_DUR):
    """Concatenate hanya segmen audio dari Participant."""
    chunks = []
    for (start, stop) in segments:
        dur = stop - start
        if dur < min_dur:
            continue
        s_idx = int(start * sr)
        e_idx = min(int(stop * sr), len(y))
        if s_idx < e_idx:
            chunks.append(y[s_idx:e_idx])
    return np.concatenate(chunks) if chunks else np.array([])


In [17]:
def get_vuv_mask_from_covarep(covarep_path, n_samples, sr=TARGET_SR, fps=100):
    """
    Membaca kolom VUV dari COVAREP.csv (kolom ke-2, index 1).
    VUV = 1 → voiced (ada suara), VUV = 0 → unvoiced/silence.
    Konversi dari frame-level ke sample-level mask boolean.

    COVAREP di-ekstrak pada 100 fps (frame setiap 10ms).
    """
    try:
        # COVAREP tidak punya header — baca sebagai numpy langsung
        covarep = np.genfromtxt(covarep_path, delimiter=',', filling_values=0)
        if covarep.ndim < 2 or covarep.shape[1] <= COVAREP_VUV_COL:
            return None

        vuv_frames = covarep[:, COVAREP_VUV_COL]  # 0=unvoiced, 1=voiced
        hop_samples = int(sr / fps)  # 160 samples per frame @ 16kHz, 100fps

        # Expand frame-level → sample-level mask
        mask = np.repeat(vuv_frames, hop_samples)
        # Trim atau pad agar panjangnya sama dengan n_samples
        if len(mask) > n_samples:
            mask = mask[:n_samples]
        else:
            mask = np.pad(mask, (0, n_samples - len(mask)), constant_values=0)

        return mask.astype(bool)
    except Exception as e:
        log.warning(f"Gagal baca COVAREP VUV: {e}")
        return None

def apply_vad_covarep(y, vuv_mask):
    """Terapkan VUV mask: ambil hanya frame voiced."""
    if vuv_mask is None or len(vuv_mask) != len(y):
        return None
    voiced = y[vuv_mask]
    return voiced if len(voiced) > 0 else None


In [18]:
def apply_vad_librosa(y, top_db=TOP_DB_VAD):
    """Fallback VAD berbasis energy (librosa) jika COVAREP tidak tersedia."""
    if len(y) == 0:
        return y
    intervals = librosa.effects.split(y, top_db=top_db,
                                       frame_length=512, hop_length=128)
    if len(intervals) == 0:
        return y
    return np.concatenate([y[s:e] for s, e in intervals])


In [19]:
def reduce_noise_audio(y, sr):
    """
    Noise reduction menggunakan library noisereduce.
    Estimasi noise dari 0.5 detik pertama (biasanya silence awal interview).
    """
    try:
        noise_sample_len = min(int(sr * 0.5), len(y) // 4)
        if noise_sample_len < 100:
            return y
        noise_clip = y[:noise_sample_len]
        reduced = nr.reduce_noise(y=y, sr=sr, y_noise=noise_clip,
                                  prop_decrease=0.8, stationary=False)
        return reduced
    except Exception as e:
        log.warning(f"Noise reduction gagal, skip: {e}")
        return y


In [20]:
def normalize_audio(y):
    """Peak amplitude normalization ke range [-1, 1]."""
    max_amp = np.max(np.abs(y))
    return y / max_amp if max_amp > 1e-8 else y


In [21]:
def preprocess_participant(pid, folder_path):
    """
    Full preprocessing pipeline untuk satu partisipan DAIC-WOZ.
    Returns dict berisi info preprocessing (status, durasi, dll).
    """
    audio_path      = os.path.join(folder_path, f"{pid}_AUDIO.wav")
    transcript_path = os.path.join(folder_path, f"{pid}_TRANSCRIPT.csv")
    covarep_path    = os.path.join(folder_path, f"{pid}_COVAREP.csv")
    output_path     = os.path.join(CLEANED_DIR, f"{pid}.wav")

    info = {
        'participant_id'   : pid,
        'status'           : 'ok',
        'dur_original_s'   : 0,
        'dur_after_diar_s' : 0,
        'dur_after_vad_s'  : 0,
        'dur_final_s'      : 0,
        'vad_method'       : 'none',
        'n_segments'       : 0,
        'output_path'      : output_path,
        'error_msg'        : '',
    }

    # ------------------------------------------------------------------
    # STEP 1: Load audio (resample + mono)
    # ------------------------------------------------------------------
    if not os.path.exists(audio_path):
        info['status'] = 'error'
        info['error_msg'] = f"File audio tidak ditemukan: {audio_path}"
        return info

    try:
        y_raw, _ = librosa.load(audio_path, sr=TARGET_SR, mono=True)
        info['dur_original_s'] = round(len(y_raw) / TARGET_SR, 2)
    except Exception as e:
        info['status'] = 'error'
        info['error_msg'] = f"Gagal load audio: {e}"
        return info

    # ------------------------------------------------------------------
    # STEP 2: Speaker Diarization via TRANSCRIPT.csv
    # Ambil hanya segmen suara Participant, buang suara Ellie (interviewer)
    # ------------------------------------------------------------------
    segments = get_participant_segments(transcript_path) if os.path.exists(transcript_path) else []
    info['n_segments'] = len(segments)

    if segments:
        y_diar = extract_segments_audio(y_raw, TARGET_SR, segments)
        if len(y_diar) == 0:
            y_diar = y_raw  # fallback
            log.warning(f"PID {pid}: Diarization menghasilkan audio kosong, gunakan audio penuh.")
    else:
        y_diar = y_raw
        log.warning(f"PID {pid}: Tidak ada TRANSCRIPT, gunakan audio penuh.")

    info['dur_after_diar_s'] = round(len(y_diar) / TARGET_SR, 2)

    # ------------------------------------------------------------------
    # STEP 3: Voice Activity Detection (VAD)
    # Prioritas: COVAREP VUV column → fallback: librosa energy VAD
    # ------------------------------------------------------------------
    y_vad = None

    if os.path.exists(covarep_path):
        vuv_mask = get_vuv_mask_from_covarep(covarep_path, len(y_diar))
        y_vad = apply_vad_covarep(y_diar, vuv_mask)
        if y_vad is not None and len(y_vad) > TARGET_SR * 1:
            info['vad_method'] = 'covarep_vuv'

    if y_vad is None or len(y_vad) < TARGET_SR * 1:
        # Fallback ke energy-based VAD
        y_vad = apply_vad_librosa(y_diar)
        info['vad_method'] = 'librosa_energy'

    if len(y_vad) == 0:
        y_vad = y_diar  # worst-case fallback
        info['vad_method'] = 'no_vad'

    info['dur_after_vad_s'] = round(len(y_vad) / TARGET_SR, 2)

    # ------------------------------------------------------------------
    # STEP 4: Noise Reduction
    # ------------------------------------------------------------------
    y_clean = reduce_noise_audio(y_vad, TARGET_SR)

    # ------------------------------------------------------------------
    # STEP 5: Amplitude Normalization
    # ------------------------------------------------------------------
    y_final = normalize_audio(y_clean)
    info['dur_final_s'] = round(len(y_final) / TARGET_SR, 2)

    # ------------------------------------------------------------------
    # STEP 6: Validasi & Simpan
    # ------------------------------------------------------------------
    if len(y_final) < TARGET_SR * 2:  # minimal 2 detik
        info['status'] = 'warning_too_short'
        info['error_msg'] = f"Audio final terlalu pendek: {info['dur_final_s']} detik"
        log.warning(f"PID {pid}: {info['error_msg']}")

    try:
        sf.write(output_path, y_final, TARGET_SR, subtype='PCM_16')
    except Exception as e:
        info['status'] = 'error'
        info['error_msg'] = f"Gagal simpan: {e}"
        return info

    log.info(
        f"PID {pid:4d} | {info['dur_original_s']:7.1f}s → "
        f"diar {info['dur_after_diar_s']:7.1f}s → "
        f"vad({info['vad_method']}) {info['dur_after_vad_s']:7.1f}s → "
        f"final {info['dur_final_s']:7.1f}s | {info['status']}"
    )
    return info


In [22]:
def scan_daic_folders(daic_dir):
    """Scan semua folder partisipan format {ID}_P di DAIC-WOZ directory."""
    folders = sorted([
        f for f in os.listdir(daic_dir)
        if os.path.isdir(os.path.join(daic_dir, f)) and f.endswith('_P')
    ])
    participants = []
    for folder in folders:
        try:
            pid = int(folder.replace('_P', ''))
            participants.append((pid, os.path.join(daic_dir, folder)))
        except ValueError:
            continue
    return participants


In [23]:
if __name__ == '__main__':
    print("\n" + "="*60)
    print(" DAIC-WOZ Audio Preprocessing Pipeline")
    print("="*60)

    participants = scan_daic_folders(DAIC_DIR)
    print(f"Total partisipan ditemukan: {len(participants)}\n")

    results = []
    failed_pids = []

    for pid, folder_path in tqdm(participants, desc="Preprocessing", unit="partisipan"):
        info = preprocess_participant(pid, folder_path)
        results.append(info)
        if 'error' in info['status']:
            failed_pids.append(pid)

    # ------------------------------------------------------------------
    # Simpan CSV log
    # ------------------------------------------------------------------
    df_log = pd.DataFrame(results)
    log_path = os.path.join(LOG_DIR, 'preprocessing_log.csv')
    df_log.to_csv(log_path, index=False)

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    df_ok = df_log[df_log['status'] == 'ok']
    df_warn = df_log[df_log['status'].str.startswith('warning', na=False)]
    df_err = df_log[df_log['status'].str.startswith('error', na=False)]

    print("\n" + "="*60)
    print(" SUMMARY PREPROCESSING")
    print("="*60)
    print(f"  Total diproses   : {len(results)}")
    print(f"  Berhasil (ok)    : {len(df_ok)}")
    print(f"  Warning          : {len(df_warn)}")
    print(f"  Gagal (error)    : {len(df_err)}")
    if failed_pids:
        print(f"  PID gagal        : {failed_pids}")

    if len(df_ok) > 0:
        print(f"\n  Statistik Durasi Audio (detik):")
        stats = df_ok[['dur_original_s', 'dur_after_diar_s', 'dur_after_vad_s', 'dur_final_s']].describe().round(1)
        print(stats.to_string())

        reduction = ((df_ok['dur_original_s'] - df_ok['dur_final_s']) / df_ok['dur_original_s'] * 100).mean()
        print(f"\n  Rata-rata pengurangan durasi : {reduction:.1f}%")

        vad_counts = df_ok['vad_method'].value_counts()
        print(f"\n  VAD Method yang digunakan:")
        for method, count in vad_counts.items():
            print(f"    {method:25s}: {count} partisipan")

    print(f"\n  Log tersimpan di : {log_path}")
    print(f"  Audio cleaned di : {CLEANED_DIR}")
    print("="*60 + "\n")


 DAIC-WOZ Audio Preprocessing Pipeline
Total partisipan ditemukan: 189



Preprocessing:   0%|          | 0/189 [00:00<?, ?partisipan/s]

PID  300 |   648.5s → diar   155.8s → vad(covarep_vuv)    73.1s → final    73.1s | ok
PID  300 |   648.5s → diar   155.8s → vad(covarep_vuv)    73.1s → final    73.1s | ok


Preprocessing:   1%|          | 1/189 [00:15<50:05, 15.99s/partisipan]

PID  301 |   823.9s → diar   475.2s → vad(covarep_vuv)   179.4s → final   179.4s | ok
PID  301 |   823.9s → diar   475.2s → vad(covarep_vuv)   179.4s → final   179.4s | ok


...

PID  349 |  1216.0s → diar   515.3s → vad(covarep_vuv)   152.5s → final   152.5s | ok
PID  349 |  1216.0s → diar   515.3s → vad(covarep_vuv)   152.5s → final   152.5s | ok


Preprocessing:  26%|██▌       | 49/189 [03:15<09:17,  3.99s/partisipan]

PID  350 |   881.8s → diar   458.2s → vad(covarep_vuv)   217.9s → final   217.9s | ok
PID  350 |   881.8s → diar   458.2s → vad(covarep_vuv)   217.9s → final   217.9s | ok


Preprocessing:  26%|██▋       | 50/189 [03:20<09:16,  4.00s/partisipan]


KeyboardInterrupt: 